In [ ]:
# предобработка признаков
import numpy as np
import pandas as pd
import sklearn as sk
from ucimlrepo import fetch_ucirepo

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier



bank = fetch_ucirepo(id=222)
X = bank.data.features.drop(columns=["duration"])
X = X.rename(columns={"day_of_week": "day"})
X["day"] = X["day"].astype(str)
y = (bank.data.targets["y"] == "yes").astype(int)

num_cols = X.select_dtypes("number").columns
cat_cols = X.select_dtypes("object").columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cat_cols),
])

#Logistic regression
logreg_model = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

# препроцессинг для деревьев: без StandardScaler
tree_preprocess = ColumnTransformer([
    ("num", "passthrough", num_cols),
    ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore"), cat_cols),
])

# Random Forest
rf_model = Pipeline([
    ("prep", tree_preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    )),
])

# XGBoost: дисбаланс классов через scale_pos_weight = n_neg / n_pos
spw = (y_train == 0).sum() / (y_train == 1).sum()

# XGBoost
xgb_model = Pipeline([
    ("prep", tree_preprocess),
    ("clf", XGBClassifier(
        n_estimators=400,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=spw,
        eval_metric="logloss",
        n_jobs=-1,
        random_state=42,
    )),
])

logreg_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
xgb_model.fit(X_train, y_train)

# Сравнение всех моделей
for name, m in [("LogReg", logreg_model), ("RF", rf_model), ("XGBoost", xgb_model)]:
    proba = m.predict_proba(X_test)[:, 1]
    pred = m.predict(X_test)
    print(f"{name:9} ROC-AUC={roc_auc_score(y_test, proba):.3f}  "
          f"PR-AUC={average_precision_score(y_test, proba):.3f}")



cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {"roc_auc": "roc_auc", "pr_auc": "average_precision"}

models = {"LogReg": logreg_model, "RF": rf_model, "XGBoost": xgb_model}
cv_results = {}

for name, m in models.items():
    res = cross_validate(m, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    cv_results[name] = res
    roc, pr = res["test_roc_auc"], res["test_pr_auc"]
    print(f"{name:9} ROC-AUC={roc.mean():.3f} ± {roc.std():.3f}  "
          f"PR-AUC={pr.mean():.3f} ± {pr.std():.3f}")

# попарное сравнение RF и XGBoost
for metric in ["test_roc_auc", "test_pr_auc"]:
    diff = cv_results["XGBoost"][metric] - cv_results["RF"][metric]
    print(f"XGBoost - RF, {metric}: mean={diff.mean():+.4f}, "
          f"по фолдам={np.round(diff, 4)}")


LogReg    ROC-AUC=0.774  PR-AUC=0.409
RF        ROC-AUC=0.799  PR-AUC=0.443
XGBoost   ROC-AUC=0.796  PR-AUC=0.456
LogReg    ROC-AUC=0.770 ± 0.001  PR-AUC=0.401 ± 0.007
RF        ROC-AUC=0.791 ± 0.004  PR-AUC=0.433 ± 0.013
XGBoost   ROC-AUC=0.796 ± 0.003  PR-AUC=0.437 ± 0.012
XGBoost - RF, test_roc_auc: mean=+0.0050, по фолдам=[-0.004   0.0081  0.0046  0.0049  0.0116]
XGBoost - RF, test_pr_auc: mean=+0.0043, по фолдам=[-0.0013  0.0024  0.0172  0.0063 -0.003 ]
